# Task 3 – Downsampling (Naive vs. Proper)

For a subset of **200 samples** test how reducing the sample rate affects authentication.

Two strategies × three factors:

| Strategy | ÷2 (→ 8 kHz) | ÷5 (→ 3.2 kHz) | ÷10 (→ 1.6 kHz) |
|---|---|---|---|
| **Naive** | keep every 2nd sample | every 5th | every 10th |
| **Proper** | torchaudio resample (anti-aliased) + upsample back | same | same |

Naive downsampling passes **fewer samples at 16 kHz** to the model (shorter audio, aliasing).
Proper downsampling preserves duration but **band-limits** the signal.

**Prerequisites:** `task1_baseline.ipynb` must have been run (`results/task1_results.json` exists).

## 0. Setup

In [ ]:
import sys, json, random
from pathlib import Path

SRC = Path("../src").resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import torch
import torchaudio
import matplotlib.pyplot as plt

from config import cfg
from embeddings import load_model, load_audio
from database import get_collection, list_enrolled
from threshold import compute_eer, compute_metrics_at_threshold, _load_enrolled_embeddings_bulk

RESULTS_DIR = cfg.paths.results_dir
TEST_DIR    = cfg.paths.test_dir
RANDOM_SEED = cfg.dataset.random_seed
SAMPLE_RATE = cfg.audio.sample_rate  # 16000

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Results → {RESULTS_DIR}")

## 1. Load Task 1 baseline

In [ ]:
task1_path = RESULTS_DIR / "task1_results.json"
if not task1_path.exists():
    raise FileNotFoundError("Run task1_baseline.ipynb first to generate task1_results.json")

with open(task1_path) as f:
    task1 = json.load(f)

eer_threshold = task1["eer_threshold"]
print(f"Task 1 baseline:")
print(f"  EER       = {task1['eer_pct']:.2f}%")
print(f"  Accuracy  = {task1['accuracy_at_eer_pct']:.2f}%")
print(f"  Threshold = {eer_threshold:.4f}")

## 2. Load model and enrolled embeddings

In [ ]:
print("Loading ECAPA-TDNN model...")
model = load_model()

collection = get_collection()
enrolled_ids = set(list_enrolled(collection))
print(f"Enrolled speakers: {len(enrolled_ids)}")

# Bulk load avoids the chromadb per-ID get bug
enrolled_cache: dict[str, np.ndarray] = _load_enrolled_embeddings_bulk(collection)
print(f"Loaded {len(enrolled_cache)} enrolled embeddings.")

if not enrolled_ids:
    raise RuntimeError("No speakers enrolled. Run src/enroll.py first.")

## 3. Select 200 test samples

Build 100 genuine + 100 impostor trials from `data/test/`.

In [ ]:
N_GENUINE  = 100
N_IMPOSTOR = 100

rng = random.Random(RANDOM_SEED)

speaker_files: dict[str, list[Path]] = {}
for spk_dir in sorted(TEST_DIR.iterdir()):
    if spk_dir.is_dir() and spk_dir.name in enrolled_ids:
        wavs = sorted(spk_dir.rglob("*.wav"))
        if wavs:
            speaker_files[spk_dir.name] = wavs

speaker_ids = sorted(speaker_files.keys())
print(f"Speakers with test files: {len(speaker_ids)}")

# Genuine pool: (test_wav, enrolled_id, is_genuine)
genuine_pool = []
for spk_id in speaker_ids:
    for wav in speaker_files[spk_id]:
        genuine_pool.append((wav, spk_id, True))
rng.shuffle(genuine_pool)

# Impostor pool: wav of speaker A tested against enrolled profile of speaker B
impostor_pool = []
for spk_id in speaker_ids:
    other_ids = [s for s in speaker_ids if s != spk_id]
    if not other_ids:
        continue
    for imp_id in rng.sample(other_ids, min(2, len(other_ids))):
        imp_wav = rng.choice(speaker_files[imp_id])
        impostor_pool.append((imp_wav, spk_id, False))
rng.shuffle(impostor_pool)

trials = genuine_pool[:N_GENUINE] + impostor_pool[:N_IMPOSTOR]
rng.shuffle(trials)

print(f"Genuine trials:  {sum(1 for t in trials if t[2])}")
print(f"Impostor trials: {sum(1 for t in trials if not t[2])}")
print(f"Total:           {len(trials)}")

## 4. Downsampling functions

- **Naive**: `waveform[:, ::factor]` — no anti-aliasing, passes fewer samples at 16 kHz to the model
- **Proper**: `torchaudio.functional.resample` — applies a low-pass filter before decimating, then upsamples back to 16 kHz

In [ ]:
# (key, display_label, method, factor)
CONDITIONS = [
    ("naive_2x",   "Naive ÷2",   "naive",  2),
    ("naive_5x",   "Naive ÷5",   "naive",  5),
    ("naive_10x",  "Naive ÷10",  "naive",  10),
    ("proper_2x",  "Proper ÷2",  "proper", 2),
    ("proper_5x",  "Proper ÷5",  "proper", 5),
    ("proper_10x", "Proper ÷10", "proper", 10),
]


def apply_naive_downsample(waveform: torch.Tensor, factor: int) -> torch.Tensor:
    """Keep every factor-th sample. No anti-aliasing filter."""
    return waveform[:, ::factor]


def apply_proper_downsample(
    waveform: torch.Tensor, factor: int, sr: int = SAMPLE_RATE
) -> torch.Tensor:
    """Anti-aliased downsample then upsample back to sr (round-trip)."""
    low_sr = sr // factor
    return torchaudio.functional.resample(
        torchaudio.functional.resample(waveform, sr, low_sr),
        low_sr,
        sr,
    )


def embed_waveform(model, waveform: torch.Tensor) -> np.ndarray | None:
    """Extract L2-normalized embedding. Returns None if waveform is too short (<10 ms)."""
    if waveform.shape[1] < 160:
        return None
    with torch.no_grad():
        emb = model.encode_batch(waveform)
    emb = emb.squeeze().cpu().numpy().astype(np.float32)
    norm = np.linalg.norm(emb)
    return emb / norm if norm > 0 else emb


print(f"Conditions defined: {[c[1] for c in CONDITIONS]}")

## 5. Compute scores for all conditions

In [ ]:
duration_stats: dict[str, list[float]] = {"original": []}
for key, *_ in CONDITIONS:
    duration_stats[key] = []

results: dict[str, dict] = {
    key: {"genuine": [], "impostor": [], "skipped": 0}
    for key, *_ in CONDITIONS
}

audio_cache: dict[Path, torch.Tensor] = {}

def get_cached(path: Path) -> torch.Tensor:
    if path not in audio_cache:
        audio_cache[path] = load_audio(path)
    return audio_cache[path]


print(f"Processing {len(trials)} trials × {len(CONDITIONS)} conditions...")

for i, (wav_path, enrolled_id, is_genuine) in enumerate(trials):
    if enrolled_id not in enrolled_cache:
        print(f"  [warn] {enrolled_id}: no enrolled embedding, skipping")
        continue

    enrolled_emb = enrolled_cache[enrolled_id]
    waveform = get_cached(wav_path)  # (1, T) at 16 kHz
    duration_stats["original"].append(waveform.shape[1] / SAMPLE_RATE)

    for key, label, method, factor in CONDITIONS:
        if method == "naive":
            w = apply_naive_downsample(waveform, factor)
        else:
            w = apply_proper_downsample(waveform, factor)

        duration_stats[key].append(w.shape[1] / SAMPLE_RATE)

        query_emb = embed_waveform(model, w)
        if query_emb is None:
            results[key]["skipped"] += 1
            continue

        score = float(np.dot(query_emb, enrolled_emb))
        bucket = "genuine" if is_genuine else "impostor"
        results[key][bucket].append(score)

    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(trials)}...")

print("Done.")

## 6. Duration analysis

Naive downsampling reduces the number of samples passed to the model,
effectively shortening the audio seen by ECAPA-TDNN.
Proper downsampling preserves duration.

In [ ]:
print(f"{'Condition':<16} {'Avg dur (s)':>12} {'Min (s)':>10} {'Max (s)':>10} {'Skipped':>8}")
print("-" * 60)

orig = duration_stats["original"]
print(f"{'Original':<16} {np.mean(orig):>12.3f} {np.min(orig):>10.3f} {np.max(orig):>10.3f}")

for key, label, method, factor in CONDITIONS:
    durs    = duration_stats[key]
    skipped = results[key]["skipped"]
    print(f"  {label:<14} {np.mean(durs):>12.3f} {np.min(durs):>10.3f} {np.max(durs):>10.3f} {skipped:>8}")

# Duration plot
fig, ax = plt.subplots(figsize=(11, 4))
cond_labels = ["Original"] + [label for _, label, _, _ in CONDITIONS]
avg_durs    = [np.mean(duration_stats["original"])] + [
    np.mean(duration_stats[key]) for key, *_ in CONDITIONS
]
bar_colors = ["gray", "#d62728", "#ff7f0e", "#8c564b", "#1f77b4", "#2ca02c", "#9467bd"]

bars = ax.bar(cond_labels, avg_durs, color=bar_colors, edgecolor="white")
ax.bar_label(bars, fmt="%.2fs", fontsize=9)
ax.axhline(avg_durs[0], color="gray", linestyle="--", linewidth=1, label="Original mean")
ax.set_ylabel("Avg duration fed to model (s)")
ax.set_title("Task 3 – Signal duration reaching ECAPA-TDNN per condition")
ax.legend()
fig.tight_layout()

out = RESULTS_DIR / "task3_duration.png"
fig.savefig(out, dpi=150)
plt.show()
print(f"Saved: {out}")

## 7. Metrics per condition

In [ ]:
rows = []
print(f"{'Condition':<15} {'N gen':>6} {'N imp':>6} {'EER%':>8} {'FAR%':>8} {'FRR%':>8} {'Acc%':>8}")
print("-" * 62)

for key, label, method, factor in CONDITIONS:
    gen = np.array(results[key]["genuine"])
    imp = np.array(results[key]["impostor"])
    if len(gen) < 2 or len(imp) < 2:
        print(f"  {label:<13}: not enough samples (skipped={results[key]['skipped']})")
        continue

    eer, _ = compute_eer(gen, imp)
    m = compute_metrics_at_threshold(gen, imp, eer_threshold)
    row = {
        "key":           key,
        "label":         label,
        "method":        method,
        "factor":        factor,
        "n_genuine":     len(gen),
        "n_impostor":    len(imp),
        "skipped":       results[key]["skipped"],
        "eer_pct":       round(eer * 100, 2),
        "far_pct":       round(m["far"] * 100, 2),
        "frr_pct":       round(m["frr"] * 100, 2),
        "accuracy_pct":  round(m["accuracy"] * 100, 2),
        "avg_duration_s": round(float(np.mean(duration_stats[key])), 3),
    }
    rows.append(row)
    print(
        f"  {label:<13} {len(gen):>6} {len(imp):>6}"
        f" {eer*100:>8.2f} {m['far']*100:>8.2f}"
        f" {m['frr']*100:>8.2f} {m['accuracy']*100:>8.2f}"
    )

print("-" * 62)
print(f"  [Task 1 baseline: EER={task1['eer_pct']:.2f}%  Acc={task1['accuracy_at_eer_pct']:.2f}%]")

## 8. Score distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
row_labels = ["Naive (no anti-aliasing)", "Proper (anti-aliased)"]

for ax_idx, (ax, (key, label, method, factor)) in enumerate(zip(axes.flat, CONDITIONS)):
    gen = np.array(results[key]["genuine"])
    imp = np.array(results[key]["impostor"])

    if len(gen) < 2 or len(imp) < 2:
        ax.text(0.5, 0.5, "Not enough data", ha="center", va="center", transform=ax.transAxes)
        continue

    ax.hist(gen, bins=40, alpha=0.65, color="steelblue", density=True, label="Genuine")
    ax.hist(imp, bins=40, alpha=0.65, color="tomato",    density=True, label="Impostor")
    ax.axvline(eer_threshold, color="black", linestyle="--", linewidth=1.2,
               label=f"thr={eer_threshold:.3f}")

    row_data = next((r for r in rows if r["key"] == key), None)
    if row_data:
        eer_f, _ = compute_eer(gen, imp)
        ax.set_title(
            f"{label}\nEER={eer_f*100:.2f}%  Acc={row_data['accuracy_pct']:.1f}%",
            fontsize=10,
        )
    ax.set_xlabel("Cosine similarity")
    ax.legend(fontsize=8)

for i, row_label in enumerate(row_labels):
    axes[i][0].set_ylabel(f"Density\n({row_label})")

fig.suptitle("Task 3 – Score distributions by downsampling condition", fontsize=13, fontweight="bold")
fig.tight_layout()

out = RESULTS_DIR / "task3_score_distributions.png"
fig.savefig(out, dpi=150)
plt.show()
print(f"Saved: {out}")

## 9. EER & Accuracy comparison with Task 1

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels   = [r["label"] for r in rows] + ["Task 1\n(baseline)"]
eer_vals = [r["eer_pct"]      for r in rows] + [task1["eer_pct"]]
acc_vals = [r["accuracy_pct"] for r in rows] + [task1["accuracy_at_eer_pct"]]

# Naive = warm tones, Proper = cool tones, Baseline = coral
palette = ["#d62728", "#ff7f0e", "#8c564b", "#1f77b4", "#2ca02c", "#9467bd", "coral"]
colors  = palette[:len(labels)]

for ax, vals, ylabel, title in [
    (axes[0], eer_vals, "EER (%)",      "EER by condition vs. Task 1"),
    (axes[1], acc_vals, "Accuracy (%)", "Accuracy by condition vs. Task 1"),
]:
    bars = ax.bar(labels, vals, color=colors, edgecolor="white", linewidth=0.5)
    ax.bar_label(bars, fmt="%.2f%%", fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

axes[0].set_ylim(0, max(eer_vals) * 1.3)
axes[1].set_ylim(max(0, min(acc_vals) - 5), 100)

fig.suptitle("Task 3 – Downsampling: Impact on System Performance", fontsize=13, fontweight="bold")
fig.tight_layout()

out = RESULTS_DIR / "task3_comparison.png"
fig.savefig(out, dpi=150)
plt.show()
print(f"Saved: {out}")

## 10. Save results

In [ ]:
summary = {
    "task": 3,
    "description": "Downsampling: naive (keep every N-th sample) vs proper (anti-aliased resample) at ÷2, ÷5, ÷10",
    "n_trials": len(trials),
    "eer_threshold_used": eer_threshold,
    "original_avg_duration_s": round(float(np.mean(duration_stats["original"])), 3),
    "per_condition": rows,
    "task1_eer_pct":      task1["eer_pct"],
    "task1_accuracy_pct": task1["accuracy_at_eer_pct"],
}

out_json = RESULTS_DIR / "task3_results.json"
with open(out_json, "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "═" * 52)
print("  TASK 3 – DOWNSAMPLING RESULTS")
print("═" * 52)
print(f"  {'Condition':<14} {'Avg dur':>8} {'EER%':>8} {'FAR%':>8} {'FRR%':>8} {'Acc%':>8}")
print("  " + "-" * 50)
for r in rows:
    print(
        f"  {r['label']:<14} {r['avg_duration_s']:>7.2f}s"
        f" {r['eer_pct']:>8.2f} {r['far_pct']:>8.2f}"
        f" {r['frr_pct']:>8.2f} {r['accuracy_pct']:>8.2f}"
    )
print("  " + "-" * 50)
orig_dur = float(np.mean(duration_stats["original"]))
print(
    f"  {'Baseline (T1)':<14} {orig_dur:>7.2f}s"
    f" {task1['eer_pct']:>8.2f} {task1['far_at_eer_pct']:>8.2f}"
    f" {task1['frr_at_eer_pct']:>8.2f} {task1['accuracy_at_eer_pct']:>8.2f}"
)
print("═" * 52)
print(f"\nJSON saved: {out_json}")